<a href="https://colab.research.google.com/github/williamng252/BT_tri_tue_nhan_tao/blob/main/BTVN_8puzzle_BFS_and_vaccum_cleaner_basemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BÀI 1: MÁY HÚT BỤI - MODEL-BASED REFLEX AGENT


In [1]:
import random

class ModelBasedVacuum:
    def __init__(self, grid):
        self.grid = grid
        self.m, self.n = len(grid), len(grid[0])
        self.x, self.y = 0, 0
        self.model = set() # Trí nhớ lưu các ô đã sạch
        self.steps = 0

    def get_valid_moves(self):
        moves = []
        if self.x > 0: moves.append("Up")
        if self.x < self.m - 1: moves.append("Down")
        if self.y > 0: moves.append("Left")
        if self.y < self.n - 1: moves.append("Right")
        return moves

    def rule_match(self, current_value, valid_moves):
        if current_value == 1:
            return "Suck"

        unvisited_moves = []
        for move in valid_moves:
            nx, ny = self.x, self.y
            if move == "Up": nx -= 1
            elif move == "Down": nx += 1
            elif move == "Left": ny -= 1
            elif move == "Right": ny += 1

            if (nx, ny) not in self.model:
                unvisited_moves.append(move)

        return random.choice(unvisited_moves) if unvisited_moves else random.choice(valid_moves)

    def execute(self):
        while True:
            # Kiểm tra phòng đã sạch hết chưa
            if all(1 not in row for row in self.grid):
                print(f"\nTHÀNH CÔNG: Dọn sạch phòng sau {self.steps} bước!")
                break

            self.steps += 1
            current_value = self.grid[self.x][self.y]

            # Cập nhật trạng thái vào Model
            if current_value == 0:
                self.model.add((self.x, self.y))

            action = self.rule_match(current_value, self.get_valid_moves())

            # Thực thi
            if action == "Suck":
                self.grid[self.x][self.y] = 0
                self.model.add((self.x, self.y))
            else:
                if action == "Up": self.x -= 1
                elif action == "Down": self.x += 1
                elif action == "Left": self.y -= 1
                elif action == "Right": self.y += 1


room = [[0, 1, 0], [0, 0, 1], [1, 0, 0]]
agent = ModelBasedVacuum(room)
agent.execute()


THÀNH CÔNG: Dọn sạch phòng sau 10 bước!


# BÀI 2: 8-PUZZLE VỚI BREADTH-FIRST SEARCH (BFS)


In [2]:
from collections import deque

# --- 1. Định nghĩa cấu trúc Node (Theo slide hình 2) ---
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state          # Trạng thái hiện tại (tuple 9 phần tử)
        self.parent = parent        # Node cha
        self.action = action        # Hành động sinh ra node này
        self.path_cost = path_cost  # Tổng chi phí từ gốc (số bước)

# --- 2. Các hàm bổ trợ ---
def get_actions(state):
    """Lấy danh sách các hành động hợp lệ (Di chuyển ô trống 0)"""
    blank_idx = state.index(0)
    row, col = divmod(blank_idx, 3)
    actions = []
    if row > 0: actions.append('Up')
    if row < 2: actions.append('Down')
    if col > 0: actions.append('Left')
    if col < 2: actions.append('Right')
    return actions

def child_node(parent_node, action):
    """Hàm CHILD-NODE: Tạo node con từ node cha và hành động"""
    state_list = list(parent_node.state)
    blank_idx = state_list.index(0)
    row, col = divmod(blank_idx, 3)

    if action == 'Up': target_idx = (row - 1) * 3 + col
    elif action == 'Down': target_idx = (row + 1) * 3 + col
    elif action == 'Left': target_idx = row * 3 + (col - 1)
    elif action == 'Right': target_idx = row * 3 + (col + 1)

    # Hoán đổi vị trí ô trống
    state_list[blank_idx], state_list[target_idx] = state_list[target_idx], state_list[blank_idx]

    return Node(
        state=tuple(state_list),
        parent=parent_node,
        action=action,
        path_cost=parent_node.path_cost + 1
    )

def print_solution(node):
    """In lộ trình từ trạng thái ban đầu đến đích"""
    path = []
    while node:
        path.append(node)
        node = node.parent
    path.reverse()

    print(f"TÌM THẤY GIẢI PHÁP TRONG {len(path)-1} BƯỚC!\n")
    for step, n in enumerate(path):
        action_msg = f"Đi '{n.action}'" if n.action else "BẮT ĐẦU"
        print(f"Bước {step} ({action_msg}):")
        for i in range(0, 9, 3):
            print(f"| {n.state[i]} {n.state[i+1]} {n.state[i+2]} |")
        print("-" * 13)

# --- 3. Hàm BFS bám sát từng dòng mã giả (Theo slide hình 1) ---
def breadth_first_search(initial_state, goal_state):
    # node <- NODE(problem.INITIAL)
    node = Node(state=initial_state)

    # if problem.GOAL-TEST(node.STATE) then return SOLUTION(node)
    if node.state == goal_state:
        return node

    # frontier <- FIFO-QUEUE()
    frontier = deque([node])
    frontier_states = {node.state} # Phụ trợ để check nhánh if nhanh hơn

    # reached <- tập rỗng
    reached = set()

    # while not EMPTY?(frontier) do
    while frontier:
        # node <- frontier.REMOVE() (Lấy node đầu tiên trong queue)
        node = frontier.popleft()
        frontier_states.remove(node.state)

        # reached <- reached U {node.STATE}
        reached.add(node.state)

        # if problem.GOAL-TEST(node.STATE) then return SOLUTION(node)
        if node.state == goal_state:
            return node

        # for each action in problem.ACTIONS(node.STATE) do
        for action in get_actions(node.state):
            # child <- CHILD-NODE(problem, node, action)
            child = child_node(node, action)

            # if child.STATE not in reached AND child not in frontier then
            if child.state not in reached and child.state not in frontier_states:
                # frontier.INSERT(child)
                frontier.append(child)
                frontier_states.add(child.state)

    # return failure
    return None


if __name__ == "__main__":
    # Đọc ma trận Start/Goal từ slide hình 2 (Ô trống = 0)
    START_STATE = (2, 8, 3, 1, 6, 4, 7, 0, 5)
    GOAL_STATE  = (1, 2, 3, 8, 0, 4, 7, 6, 5)

    print("Đang tìm kiếm bằng BFS... (Vui lòng chờ vài giây)\n")
    solution = breadth_first_search(START_STATE, GOAL_STATE)

    if solution:
        print_solution(solution)
    else:
        print("Không tìm thấy đường đi!")

Đang tìm kiếm bằng BFS... (Vui lòng chờ vài giây)

TÌM THẤY GIẢI PHÁP TRONG 5 BƯỚC!

Bước 0 (BẮT ĐẦU):
| 2 8 3 |
| 1 6 4 |
| 7 0 5 |
-------------
Bước 1 (Đi 'Up'):
| 2 8 3 |
| 1 0 4 |
| 7 6 5 |
-------------
Bước 2 (Đi 'Up'):
| 2 0 3 |
| 1 8 4 |
| 7 6 5 |
-------------
Bước 3 (Đi 'Left'):
| 0 2 3 |
| 1 8 4 |
| 7 6 5 |
-------------
Bước 4 (Đi 'Down'):
| 1 2 3 |
| 0 8 4 |
| 7 6 5 |
-------------
Bước 5 (Đi 'Right'):
| 1 2 3 |
| 8 0 4 |
| 7 6 5 |
-------------


link git:https://github.com/williamng252/BT_tri_tue_nhan_tao.git